# 05 频域描述子与多特征联合可视化

内容脉络：
1. 频域描述子（谱质心、谱带宽、谱滚降、谱平坦度、谱对比度、频带能量占比）
2. 四类音频上的特征对比
3. 10 维帧级特征向量 → PCA → 二维散点图（不训练分类器）
4. log-mel 片段级统计池化基线

## 1. 环境自检与配置

In [ ]:
import sys
assert sys.version_info >= (3, 9), "需要 Python 3.9+"

import numpy as np
import matplotlib.pyplot as plt

# 配置 matplotlib 中文字体（macOS/Windows/Linux）
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['PingFang HK', 'STHeiti', 'Heiti TC', 'Arial Unicode MS', 'Hiragino Sans GB', 'Microsoft YaHei', 'SimHei', 'Noto Sans CJK SC', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False  # 解决负号显示为方框的问题

from pathlib import Path
import warnings

import librosa
import librosa.display

SAMPLE_RATE = 22050

# 路径推断：从 cwd 向上找含 CODE/datasets 的目录
_p = Path.cwd()
while not (_p / "CODE" / "datasets").exists():
    _parent = _p.parent
    if _parent == _p:
        raise FileNotFoundError("未找到项目根目录（包含 CODE/datasets 的目录），请在项目内运行本 Notebook")
    _p = _parent
BASE_DIR = _p
DATASET_DIR = BASE_DIR / "CODE" / "datasets" / "audio_author"
OUTPUT_FIG_DIR = BASE_DIR / "CODE" / "chapter05" / "output_figures"
OUTPUT_FIG_DIR.mkdir(exist_ok=True)

print(f"librosa={librosa.__version__}")

## 2. 加载四类音频并计算 STFT

这些频域描述子都基于 STFT 幅度谱计算，因此先统一算出频谱。

In [ ]:
audio_sources = {
    "钢琴": (DATASET_DIR / "piano_solo.wav", 10.0),
    "打击乐": (DATASET_DIR / "orch_perc.wav", 2.0),
    "小提琴": (DATASET_DIR / "zhao_violin_wet.wav", 15.0),
    "人声": (DATASET_DIR / "xiaohetang_vox.wav", 30.0),
}

N_FFT = 2048
HOP_LENGTH = 512

spectrograms = {}
for label, (path, start) in audio_sources.items():
    samples, sr = librosa.load(path, sr=SAMPLE_RATE, mono=True, offset=start, duration=3.0)
    stft = librosa.stft(samples, n_fft=N_FFT, hop_length=HOP_LENGTH, window="hann")
    magnitude = np.abs(stft)
    log_spec = librosa.amplitude_to_db(magnitude, ref=np.max)
    time_frames = librosa.frames_to_time(np.arange(magnitude.shape[1]), sr=SAMPLE_RATE, hop_length=HOP_LENGTH)
    spectrograms[label] = {
        "samples": samples,
        "magnitude": magnitude,
        "log_spec": log_spec,
        "time_frames": time_frames,
    }
    print(f"{label:12s}: STFT shape = {magnitude.shape}")

## 3. 六类频域描述子

| 特征 | 直观含义 | 输出形状 |
|:---|:---|:---|
| 谱质心 | 幅度谱的加权平均频率 | `(1, T)` |
| 谱带宽 | 围绕质心的加权离散程度 | `(1, T)` |
| 谱滚降 | 累计谱量达到指定比例的位置 | `(1, T)` |
| 谱平坦度 | 几何均值 / 算术均值 | `(1, T)` |
| 谱对比度 | 每个子带的峰谷差 | 默认 `(7, T)`，因为 `n_bands=6` 再加最低频段 |
| 频带能量占比 | 预先指定频带内的功率比例 | 每个频带 `(T,)` |

频带边界没有普遍标准，下面的 2 kHz / 8 kHz 只用于当前图示。谱对比度完整保留 7 行，绘图时另取其跨子带均值作为一条摘要曲线。


In [ ]:
def compute_band_energy_fractions(magnitude, sr, n_fft, split_low=2000, split_high=8000):
    """计算低、中、高三个预设频带的功率占比；边界需按任务验证。"""
    freq_bins = np.fft.rfftfreq(n_fft, 1 / sr)
    low_mask = freq_bins < split_low
    mid_mask = (freq_bins >= split_low) & (freq_bins < split_high)
    high_mask = freq_bins >= split_high

    power = magnitude ** 2
    low_energy = np.sum(power[low_mask], axis=0)
    mid_energy = np.sum(power[mid_mask], axis=0)
    high_energy = np.sum(power[high_mask], axis=0)
    total = low_energy + mid_energy + high_energy
    valid = total > 1e-12
    fractions = [
        np.divide(value, total, out=np.zeros_like(total), where=valid)
        for value in (low_energy, mid_energy, high_energy)
    ]
    # 对总功率不超过 1e-12 的帧，将三个频带比例置为 0；这些值不表示实际能量分布。
    return tuple(fractions)

features = {}
for label, spec in spectrograms.items():
    mag = spec["magnitude"]
    t = spec["time_frames"]

    centroid = librosa.feature.spectral_centroid(S=mag, sr=SAMPLE_RATE)[0]
    bandwidth = librosa.feature.spectral_bandwidth(S=mag, sr=SAMPLE_RATE)[0]
    rolloff = librosa.feature.spectral_rolloff(S=mag, sr=SAMPLE_RATE)[0]
    flatness = librosa.feature.spectral_flatness(S=mag)[0]
    contrast = librosa.feature.spectral_contrast(
        S=mag, sr=SAMPLE_RATE, n_bands=6
    )
    low_fraction, mid_fraction, high_fraction = compute_band_energy_fractions(
        mag, SAMPLE_RATE, N_FFT
    )

    assert contrast.shape[0] == 7
    features[label] = {
        "time": t,
        "centroid": centroid,
        "bandwidth": bandwidth,
        "rolloff": rolloff,
        "flatness": flatness,
        "contrast": contrast,
        "contrast_mean": contrast.mean(axis=0),
        "low_fraction": low_fraction,
        "mid_fraction": mid_fraction,
        "high_fraction": high_fraction,
    }
    print(f"{label:12s}: spectral_contrast shape = {contrast.shape}")


In [ ]:
fig, axes = plt.subplots(6, 4, figsize=(16, 16), sharex=True)
feature_names = [
    "谱质心 (Hz)", "谱带宽 (Hz)", "谱滚降 (Hz)",
    "谱平坦度", "谱对比度跨子带均值 (dB)", "高频能量占比"
]
feature_keys = [
    "centroid", "bandwidth", "rolloff",
    "flatness", "contrast_mean", "high_fraction"
]

for col, (label, feat) in enumerate(features.items()):
    for row, (fname, fkey) in enumerate(zip(feature_names, feature_keys)):
        ax = axes[row, col]
        ax.plot(feat["time"], feat[fkey], lw=1.2, color="0.2")
        if row == 0:
            ax.set_title(label, fontsize=12, fontweight="bold")
        if col == 0:
            ax.set_ylabel(fname, fontsize=9)
        ax.set_xlabel("时间 (s)", fontsize=9)
        ax.set_xlim(feat["time"][0], feat["time"][-1])

plt.suptitle("四段项目内录音的频域描述子（片段级观察）", fontsize=14)
plt.tight_layout()
plt.savefig(OUTPUT_FIG_DIR / "spectral_scalar_features.png", dpi=600, bbox_inches="tight")
plt.show()


**观察口径**：这些曲线只描述四个选定的 3 秒片段。可以比较同一片段内瞬态、持续段与噪声段的同步变化，也可以检查多个描述子是否给出一致线索；不能把当前数值范围当作四种乐器类别的普遍统计规律。

特别注意：理论上完全平坦的非零谱，其线性平坦度为 1（换算为 0 dB）；有限样本白噪声的 periodogram 会随机起伏，实测值取决于窗、帧长和谱估计方式。librosa 对全零静音帧施加数值地板后也会返回 1（0 dB），因此解释平坦度前应先用能量门限屏蔽静音。这里的“频带能量”是给定窗函数、STFT 缩放与离散 bin 下的谱功率和，不是经声学标定的物理能量。


## 4. 选定片段的帧级联合特征与 PCA

每个时间帧使用 10 维向量：RMS、ZCR、谱质心、谱带宽、谱滚降、谱平坦度、MFCC $c_1/c_2$、Chroma C 分量和 Chroma G 分量。

这里把 C 与 G 当作两个连续变量，避免将“最高音级索引”当作线性数值；音级索引具有环形结构，B 与 C 相邻却会被数字 11 和 0 错误地放在两端。PCA 图只展示四个选定片段的帧云，不是乐器分类性能评测。


In [ ]:
def extract_frame_feature_vector(magnitude, samples, sr, n_fft, hop_length):
    """为每帧提取 10 维连续特征，并统一使用两侧零填充的居中帧。"""
    # librosa 的 RMS 默认零填充，而 ZCR 默认复制边界值。显式零填充后以 center=False 计算，确保两者与 center=True 的 STFT 使用同一批样本。
    pad = n_fft // 2
    samples_padded = np.pad(samples, (pad, pad), mode="constant")
    rms = librosa.feature.rms(
        y=samples_padded, frame_length=n_fft, hop_length=hop_length, center=False
    )[0]
    zcr = librosa.feature.zero_crossing_rate(
        y=samples_padded, frame_length=n_fft, hop_length=hop_length, center=False
    )[0]

    centroid = librosa.feature.spectral_centroid(S=magnitude, sr=sr)[0]
    bandwidth = librosa.feature.spectral_bandwidth(S=magnitude, sr=sr)[0]
    rolloff = librosa.feature.spectral_rolloff(S=magnitude, sr=sr)[0]
    flatness = librosa.feature.spectral_flatness(S=magnitude)[0]

    # 从波形显式计算 mel 前端；不能把线性频谱的 log-power 直接冒充 log-mel。
    mfcc = librosa.feature.mfcc(
        y=samples, sr=sr, n_mfcc=3,
        n_fft=n_fft, hop_length=hop_length, n_mels=128
    )
    mfcc1, mfcc2 = mfcc[1], mfcc[2]

    chroma = librosa.feature.chroma_stft(
        y=samples, sr=sr, n_fft=n_fft, hop_length=hop_length
    )
    chroma_c = chroma[0]
    chroma_g = chroma[7]

    n_frames = min(
        magnitude.shape[1], len(rms), len(zcr), len(mfcc1), chroma.shape[1]
    )
    return np.vstack([
        rms[:n_frames], zcr[:n_frames],
        centroid[:n_frames], bandwidth[:n_frames],
        rolloff[:n_frames], flatness[:n_frames],
        mfcc1[:n_frames], mfcc2[:n_frames],
        chroma_c[:n_frames], chroma_g[:n_frames]
    ]).T

feature_vectors = {}
for label, spec in spectrograms.items():
    mat = extract_frame_feature_vector(
        spec["magnitude"], spec["samples"], SAMPLE_RATE, N_FFT, HOP_LENGTH
    )
    feature_vectors[label] = mat
    print(f"{label:12s}: feature matrix shape = {mat.shape}")


In [ ]:
# 合并四个选定片段的所有帧并做 PCA
all_vectors = []
all_labels = []
for label, mat in feature_vectors.items():
    all_vectors.append(mat)
    all_labels.extend([label] * len(mat))

X = np.vstack(all_vectors)
labels = np.array(all_labels)

X_mean = np.mean(X, axis=0)
X_std = np.std(X, axis=0) + 1e-10
X_norm = (X - X_mean) / X_std

_, singular_values, Vt = np.linalg.svd(X_norm, full_matrices=False)
components = Vt[:2]
X_pca = X_norm @ components.T
explained_variance = (singular_values ** 2) / (len(X_norm) - 1)
explained_variance_ratio = explained_variance / np.sum(explained_variance)

print(f"PCA 解释方差比：PC1 = {explained_variance_ratio[0]:.3f}, PC2 = {explained_variance_ratio[1]:.3f}")
print(f"累计解释方差 = {np.sum(explained_variance_ratio[:2]):.3f}")

colors = {"钢琴": "0.15", "打击乐": "0.35", "小提琴": "0.55", "人声": "0.75"}
markers = {"钢琴": "o", "打击乐": "s", "小提琴": "^", "人声": "D"}

fig, ax = plt.subplots(figsize=(8, 7))
for label in feature_vectors:
    mask = labels == label
    ax.scatter(
        X_pca[mask, 0], X_pca[mask, 1],
        c=colors[label], marker=markers[label],
        alpha=0.6, s=25, edgecolors="none", label=label
    )

ax.set_xlabel(f"主成分 1（{explained_variance_ratio[0]*100:.1f}% 方差）")
ax.set_ylabel(f"主成分 2（{explained_variance_ratio[1]*100:.1f}% 方差）")
ax.set_title("四个选定片段的帧级特征 → PCA（无监督投影）")
ax.legend(loc="best", markerscale=2)
ax.axhline(0, color="0.65", ls="--", lw=0.5)
ax.axvline(0, color="0.65", ls="--", lw=0.5)

plt.tight_layout()
plt.savefig(OUTPUT_FIG_DIR / "case_b_pca_scatter.png", dpi=600, bbox_inches="tight")
plt.show()


**如何解读**：若某些片段帧云在这两个主成分上分离，只能说明当前切片、当前 10 维特征与当前标准化方式记录了差异。少数极端点还可能受切片边界的居中零填充或低能量帧影响；正式分析应显式屏蔽或单独检查这些帧。点云重叠也不等于类别不可分。要讨论“乐器类别聚类”或分类边界，需要多曲目、多录音条件、独立划分和相应评价指标。


## 5. log-mel 片段级统计池化基线

前面的 PCA 例子把 RMS、ZCR、谱形、MFCC、Chroma 等手工特征拼成向量。现代系统也常改用预训练音频编码器，例如 CLAP、MERT 等。

不同预训练模型的接口并不完全相同。CLAP 通常把音频片段与文本映射到可比较的 clip-level embedding；MERT 编码器则输出随时间排列的隐藏表示，下游任务可再做池化或逐帧预测。学习表示的单维通常没有“谱质心”这类直接物理含义，其相似性语义也取决于训练目标与数据覆盖。

为明确区分固定统计量与学习表示，下面计算一个不含可学习参数的基线：对每段音频的 log-mel 各频带分别取时间均值和标准差，再拼成长度为 `2 * n_mels` 的向量。该向量是片段级统计摘要，不是 CLAP、MERT 或其他预训练编码器的输出。


In [ ]:
def compute_logmel_statistical_baseline(samples, sr, n_mels=64, n_fft=2048, hop_length=512):
    """计算各 mel 频带 log-power 的时间均值与标准差，并拼接为片段级向量。"""
    mel_power = librosa.feature.melspectrogram(
        y=samples,
        sr=sr,
        n_fft=n_fft,
        hop_length=hop_length,
        n_mels=n_mels,
        power=2.0,
    )
    logmel = librosa.power_to_db(mel_power, ref=np.max)
    return np.concatenate([logmel.mean(axis=1), logmel.std(axis=1)])

baseline_vectors = {}
for label, spec in spectrograms.items():
    baseline_vectors[label] = compute_logmel_statistical_baseline(spec["samples"], SAMPLE_RATE)

for label, vector in baseline_vectors.items():
    print(f"{label:12s}: log-mel 统计基线 shape = {vector.shape}")

print("\n该向量由固定 log-mel 统计量构成，不是 CLAP、MERT 等预训练模型的表示。")


## 6. 小结

1. `spectral_contrast(n_bands=6)` 返回 7 个子带行；图中只把跨子带均值作为摘要曲线。
2. 频带能量边界是任务参数，不存在跨任务通用的固定切分。
3. MFCC 使用 mel 前端；PCA 的 Chroma 输入使用连续的 C/G 分量，不把环形音级索引当线性数。
4. PCA 只描述四个选定片段的帧云，不支持乐器类别的泛化结论。
5. log-mel 均值/标准差池化是固定统计基线，不等同于预训练音频嵌入。


In [ ]:
print("本 Notebook 生成的图像文件：")
for prefix in ["spectral_scalar", "case_b"]:
    for f in OUTPUT_FIG_DIR.glob(f"{prefix}*.png"):
        size_kb = f.stat().st_size / 1024
        print(f"  {f.name:45s} {size_kb:8.1f} KB")